In [9]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Change DATASET to one of:
#   "DL_similarity"   →  BPIC_test_DL_similarity.csv
#   "MAE_rrt"         →  BPIC_test_MAE_rrt_minutes.csv
#   "MAE_ttne"        →  BPIC_test_MAE_ttne_minutes.csv

DATA = "BPICDA"
TARGET = "DL"
CAPPED = True

# Target column name matches the file suffix
TARGET_COLS = {
    "DL": "test_DL_similarity",
    "rrt": "test_MAE_rrt_minutes",
    "ttne": "test_MAE_ttne_minutes",
}

# Hyperparameter columns (same in all three files)
FEATURE_COLS = ["d_model", "d_ff_multiplier", "num_layers",
                "learning_rate", "dropout", "weight_decay"]

# Columns to log-transform (span multiple orders of magnitude)
LOG_COLS = ["learning_rate", "weight_decay"]

N_TREES = 48
RANDOM_STATE = 42
# ──────────────────────────────────────────────────────────────────────────────
print(f"Active dataset : {DATA}")
print(f"Target column  : {TARGET_COLS[TARGET]}")

Active dataset : BPICDA
Target column  : test_DL_similarity


In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score

# ── Load data ─────────────────────────────────────────────────────────────────
if CAPPED == True:
    df = pd.read_csv(f"{DATA}_{TARGET}_capped.csv")
else:
    df = pd.read_csv(f"{DATA}_{TARGET}.csv")
target_col = TARGET_COLS[TARGET]

X = df[FEATURE_COLS].copy()
y = df[target_col].copy()

cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)

print(f"Samples: {len(df)}   Features: {X.shape[1]}   Target: {target_col}")
print(f"\nFeature ranges:")
print(X.agg(["min", "max"]).T.to_string())


Samples: 100   Features: 6   Target: test_DL_similarity

Feature ranges:
                      min        max
d_model          8.000000  96.000000
d_ff_multiplier  2.000000   8.000000
num_layers       2.000000   8.000000
learning_rate    0.000010   0.009808
dropout          0.006696   0.695759
weight_decay     0.000011   0.009418


In [11]:
# ── Model A: Raw features (no log transform) ──────────────────────────────────
rf_raw = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=3, random_state=RANDOM_STATE)
rf_raw.fit(X, y)

RandomForestRegressor(min_samples_leaf=3, n_estimators=48, random_state=42)

In [12]:
# ── Model B: Log-transformed learning_rate & weight_decay ─────────────────────
X_log = X.copy()
for col in LOG_COLS:
    X_log[col] = np.log10(X_log[col])

rf_log = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=3, random_state=RANDOM_STATE)
rf_log.fit(X_log, y)

RandomForestRegressor(min_samples_leaf=3, n_estimators=48, random_state=42)

In [13]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score

loo = LeaveOneOut()

# LOO predictions (each point predicted from a model trained on the other 64)
y_pred_raw = cross_val_predict(rf_raw, X,     y, cv=loo)
y_pred_log = cross_val_predict(rf_log, X_log, y, cv=loo)

# R² scores
r2_train_raw = r2_score(y, rf_raw.predict(X))
r2_train_log = r2_score(y, rf_log.predict(X_log))
r2_loo_raw   = r2_score(y, y_pred_raw)
r2_loo_log   = r2_score(y, y_pred_log)

print(f"{'':30s}  {'Raw':>8}  {'Log-transformed':>15}")
print("-" * 58)
print(f"{'Train R²':30s}  {r2_train_raw:8.4f}  {r2_train_log:15.4f}")
print(f"{'LOO R²':30s}  {r2_loo_raw:8.4f}  {r2_loo_log:15.4f}")
print(f"{'Overfit gap (Train - LOO)':30s}  {r2_train_raw - r2_loo_raw:8.4f}  {r2_train_log - r2_loo_log:15.4f}")


                                     Raw  Log-transformed
----------------------------------------------------------
Train R²                          0.8270           0.8264
LOO R²                            0.4756           0.4726
Overfit gap (Train - LOO)         0.3513           0.3538
